<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


## Leveraging Apache Spark for Smart Building HVAC Monitoring

**Estimated time needed: 30 minutes**

### Objectives

After completing this lab, you will be able to:

- Explain the distributed architecture of Spark in the context of smart building monitoring
- Simulate real-time sensor data for HVAC systems in a building
- Perform SQL queries to detect critical environmental conditions and calculate average readings
- Determine the aggregated results to the console for immediate insights into room conditions


## Background
Smart Building Solutions, Inc. specializes in optimizing HVAC (heating, ventilation, and air conditioning) systems to enhance comfort and energy efficiency in commercial buildings. By monitoring temperature and humidity levels in real-time across various rooms, the company aims to ensure optimal indoor conditions and preemptively address potential HVAC issues.

With a continuous influx of sensor data, Smart Building Solutions needs to process and analyze this data in real-time to maintain the quality of the indoor environment.

## Data set description
The simulated data set comprises:

`room_id`: Unique identifier for each room (e.g., R001, R002).

`temperature`: Current temperature reading from the sensor (in °C).

`humidity`: Current humidity level reading from the sensor (in %).

`timestamp`: Time when the reading was recorded (automatically generated by Spark).
The data is generated at a rate of 5 rows per second, simulating multiple rooms with various environmental conditions.


## Challenges
Monitoring indoor environmental conditions poses several challenges:

**High data velocity**: Continuous data from multiple sensors can overwhelm traditional systems.

**Need for immediate alerts**: Delays in identifying critical conditions can lead to discomfort or system inefficiencies.

**Need for data aggregation and analysis**: Efficiently aggregating and analyzing real-time data for proactive maintenance and optimization is essential.

## Apache Spark with structured streaming
To address these challenges, Apache Spark is employed for its powerful distributed computing capabilities, enabling real-time data processing and analytics.


In [1]:
!pip install pyspark==3.1.2 -q
!pip install findspark -q

In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# FindSpark simplifies the process of using Apache Spark with Python

import findspark
findspark.init()

#import functions/Classes for sparkml

from pyspark.ml.clustering import KMeans


from pyspark.sql import SparkSession


### Set up the Spark session:


In [3]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Smart Building HVAC Monitoring") \
    .getOrCreate()


26/04/28 06:31:24 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Simulate sensor data:

Use Spark’s rate source to generate continuous readings from multiple rooms.


In [4]:
from pyspark.sql.functions import expr, rand,when

# Simulate sensor data with room IDs and readings
sensor_data = spark.readStream.format("rate").option("rowsPerSecond", 5).load() \
    .withColumn("room_id", expr("CAST(value % 10 AS STRING)")) \
    .withColumn("temperature", when(expr("value % 10 == 0"), 15)  # Set temperature to 15 for one specific record
                .otherwise(20 + rand() * 25)) \
    .withColumn("humidity", expr("40 + rand() * 30"))

### Create a temporary SQL view:

Create temporary SQL view to perform SQL queries on the streaming data.


In [7]:
# Create a temporary SQL view for the sensor data
sensor_data.createOrReplaceTempView("sensor_table")


### Define SQL queries for aggregation and analysis:

* **Critical temperature query**: Detect rooms with critical temperature levels
* **Average readings query**: Calculate average readings over a 1-minute window
* **Attention needed query**: Identify rooms that need immediate attention based on humidity levels


In [8]:
# SQL Query to detect rooms with critical temperatures
critical_temperature_query = """
    SELECT 
        room_id, 
        temperature, 
        humidity, 
        timestamp 
    FROM sensor_table 
    WHERE temperature < 18 OR temperature > 60
"""

# SQL Query to calculate average readings over a 1-minute window
average_readings_query = """
    SELECT 
        room_id, 
        AVG(temperature) AS avg_temperature, 
        AVG(humidity) AS avg_humidity, 
        window.start AS window_start 
    FROM sensor_table
    GROUP BY room_id, window(timestamp, '1 minute')
"""

# SQL Query to find rooms that need immediate attention based on humidity
attention_needed_query = """
    SELECT 
        room_id, 
        COUNT(*) AS critical_readings 
    FROM sensor_table 
    WHERE humidity < 45 OR humidity > 75
    GROUP BY room_id
"""


### Execute the SQL queries:

Execute each SQL query to create streaming DataFrames.


In [9]:
# Execute the critical temperature query
critical_temperatures_stream = spark.sql(critical_temperature_query)


# Execute the average readings query
average_readings_stream = spark.sql(average_readings_query)

# Execute the attention needed query
attention_needed_stream = spark.sql(attention_needed_query)






### Output the results to the console:

Display the results from each query in real-time.


In [10]:
# Output the results to the console for all queries
critical_query = critical_temperatures_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .queryName("Critical Temperatures") \
    .start()

average_query = average_readings_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Average Readings") \
    .start()

attention_query = attention_needed_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Attention Needed") \
    .start()



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------


[Stage 2:>                (8 + 8) / 200][Stage 4:>                (0 + 0) / 200]

+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|  69.0406097754063|2026-04-28 06:32:...|
|      0|       15.0|47.784870707208206|2026-04-28 06:32:...|
|      0|       15.0|49.541202382130976|2026-04-28 06:32:...|
|      0|       15.0|43.225751918118036|2026-04-28 06:32:...|
+-------+-----------+------------------+--------------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
+-------+-----------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|44.133768016671866|2026-04-28 06:32:...|
|      0|       15.0|  61.9888527316231|2026-04-28 06:32:...|
|      0|       15.0|52.734044063049964|2026-04-28 06:32:...|
|      0|       15.0| 53.88744272329187|2026-04-28 06:32:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---------------+------------+------------+
|room_id|avg_temperature|avg_humidity|window_start|
+-------+---------------+------------+------------+
+-------+---------------+------------+------------+



[Stage 7:> (38 + 8) / 200][Stage 8:>    (0 + 0) / 8][Stage 9:>    (0 + 0) / 8]8]

### Keep the streams running:

Ensure that the streaming queries continue to run to process incoming data.


In [ ]:
# Keep the streams running

print("********Critical Temperature Values*******")
critical_query.awaitTermination()

print("********Average Readings Values********")
average_query.awaitTermination()
print("********Attention Needed Values********")
attention_query.awaitTermination()


[Stage 7:> (40 + 8) / 200][Stage 8:>    (0 + 0) / 8][Stage 9:>    (0 + 0) / 8]

********Critical Temperature Values*******


-------------------------------------------
Batch: 3
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 63.86527723426903|2026-04-28 06:32:...|
|      0|       15.0| 40.06317515199362|2026-04-28 06:32:...|
|      0|       15.0|45.858335545544556|2026-04-28 06:32:...|
|      0|       15.0| 52.02872588526887|2026-04-28 06:32:...|
|      0|       15.0|55.693799701321936|2026-04-28 06:33:...|
|      0|       15.0| 54.25849771372646|2026-04-28 06:32:...|
|      0|       15.0| 68.82413460710197|2026-04-28 06:32:...|
|      0|       15.0| 61.37746305797608|2026-04-28 06:32:...|
|      0|       15.0| 48.55675666335228|2026-04-28 06:32:...|
|      0|       15.0| 60.31241585802616|2026-04-28 06:33:...|


-------------------------------------------
Batch: 4
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 42.09833250750481|2026-04-28 06:33:...|
|      0|       15.0| 52.20699470401121|2026-04-28 06:33:...|
|      0|       15.0| 63.97237765625418|2026-04-28 06:33:...|
|      0|       15.0| 44.21121988517909|2026-04-28 06:33:...|
|      0|       15.0| 61.22887885560988|2026-04-28 06:33:...|
|      0|       15.0|  51.5853581641396|2026-04-28 06:33:...|
|      0|       15.0|43.557792510694156|2026-04-28 06:33:...|
|      0|       15.0| 65.18802067295123|2026-04-28 06:33:...|
|      0|       15.0|59.308235276450105|2026-04-28 06:33:...|
|      0|       15.0| 48.81964566350811|2026-04-28 06:33:...|
+-------+-----------+------------------+--------------------+

----------------------------------

-------------------------------------------
Batch: 5
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 69.70468632945897|2026-04-28 06:33:...|
|      0|       15.0| 40.43447623137351|2026-04-28 06:33:...|
|      0|       15.0| 42.32217722427893|2026-04-28 06:33:...|
|      0|       15.0| 56.84025547030473|2026-04-28 06:33:...|
|      0|       15.0|  68.4029526469598|2026-04-28 06:33:...|
|      0|       15.0|61.132640432025426|2026-04-28 06:33:...|
|      0|       15.0| 43.12470460176493|2026-04-28 06:33:...|
|      0|       15.0| 51.36078676798785|2026-04-28 06:33:...|
|      0|       15.0| 44.74942738766411|2026-04-28 06:33:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 2
-------------------------------------------

...

-------------------------------------------
Batch: 7
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 41.05730691104828|2026-04-28 06:34:...|
|      0|       15.0| 66.99481766052371|2026-04-28 06:34:...|
|      0|       15.0| 66.52098889422494|2026-04-28 06:34:...|
|      0|       15.0| 64.38512553894867|2026-04-28 06:34:...|
|      0|       15.0| 59.11087937692368|2026-04-28 06:34:...|
|      0|       15.0|61.805708877020514|2026-04-28 06:34:...|
|      0|       15.0| 56.75082088118759|2026-04-28 06:34:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
|      7|                9|
|      3|  

-------------------------------------------
Batch: 8
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 68.35701836021754|2026-04-28 06:34:...|
|      0|       15.0|57.465189146584606|2026-04-28 06:34:...|
|      0|       15.0| 65.98455294310266|2026-04-28 06:34:...|
|      0|       15.0| 42.02859397181607|2026-04-28 06:34:...|
|      0|       15.0|44.577558131028844|2026-04-28 06:34:...|
|      0|       15.0| 53.26080315935937|2026-04-28 06:34:...|
|      0|       15.0|46.095138036788924|2026-04-28 06:34:...|
|      0|       15.0| 51.04049178152687|2026-04-28 06:34:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-------+------------------+------------------+--------------

-------------------------------------------
Batch: 9
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|55.638059068507076|2026-04-28 06:34:...|
|      0|       15.0| 55.57360317961141|2026-04-28 06:34:...|
|      0|       15.0| 59.35412079975329|2026-04-28 06:34:...|
|      0|       15.0| 48.82086505649751|2026-04-28 06:34:...|
|      0|       15.0|  62.3023658382308|2026-04-28 06:34:...|
|      0|       15.0| 67.17112508419171|2026-04-28 06:34:...|
|      0|       15.0|49.290416743065435|2026-04-28 06:34:...|
|      0|       15.0| 58.62092482826724|2026-04-28 06:34:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+----

-------------------------------------------
Batch: 10
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 58.82720045499714|2026-04-28 06:34:...|
|      0|       15.0|46.208362142061375|2026-04-28 06:35:...|
|      0|       15.0| 49.07071192791172|2026-04-28 06:34:...|
|      0|       15.0|47.565309456193546|2026-04-28 06:34:...|
|      0|       15.0| 67.24701824184848|2026-04-28 06:34:...|
|      0|       15.0| 48.52418079290622|2026-04-28 06:34:...|
|      0|       15.0| 69.52294573751928|2026-04-28 06:35:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-------+------------------+------------------+-------------------+
|room_id|   avg_temperature|      avg_humidity|       

[Stage 31:(138 + 8) / 200][Stage 32:>   (0 + 0) / 8][Stage 33:>   (0 + 0) / 8]8]

### Conclusion
In this lab, you explored the use of Apache Spark in smart building monitoring, particularly focusing on HVAC (heating, ventilation, and air conditioning) systems. You now understand the Spark's distributed architecture. You also understand how to simulate real-time sensor data for temperature and humidity, execute SQL queries to identify critical environmental conditions, and output aggregated results for immediate insights.


## Author(s)

Lakshmi Holla

## Other Contributors
Malika Singla
